In [ ]:
!pip install gdown -q

import gdown
import pandas as pd

url = f'https://drive.google.com/uc?id=1lklqBJlQyQ_hZRpWev-XNL8CTTzeyzmM'
output = 'cleaned2.csv'

gdown.download(url, output, quiet=False)
df = pd.read_csv(output)

Делим на train/val/test в порядке даты

In [2]:
n_tr = 10000
n_va = 11000
n_te = 12000

df_tr = df.iloc[:n_tr].copy()
df_val = df.iloc[n_tr:n_va].copy()
df_te = df.iloc[n_va: n_te] . copy()


In [3]:
len(df_tr), len(df_val), len(df_te)

(10000, 1000, 1000)

Сделаем LabelEncoding топиков для удобства

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_tr['topic'] = le.fit_transform(df_tr['topic'])
df_val['topic'] = le.transform(df_val['topic'])
df_te['topic'] = le.transform (df_te['topic'])
class_names = le.classes_

In [ ]:
class_names

array(['Культура', 'Мир', 'Россия', 'Спорт', 'Экономика'], dtype=object)

In [ ]:
df_tr.head()

,topic,textt
0,2,"Космонавты сомневаются в надежности ""Мира"" Как..."
1,1,Референдум по вопросу о самоопределении Восточ...
2,1,Литва засудила участников переворота 91 года Р...
3,1,Киргизия ведет бои на границах с Таджикистаном...
4,1,Российские национал-большевики убирают террито...


## Dataset и токенизаия

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
print(tokenizer('Hello! i Liza'))

{'input_ids': [101, 31690, 106, 248, 56597, 233, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [ ]:
max_len=256
from torch.utils.data import Dataset

class MyDataset(Dataset ):
  def __init__(self, tokenizer, texts, labels, max_len):
    self.texts = texts.to_numpy()
    self.labels=labels.to_numpy()
    self.tokenizer=tokenizer
    self.max_len = max_len

  def __len__(self, ):
     return len(self.texts)

  def __getitem__(self, idx):
    label = self.labels[idx]
    text = self.texts[idx]
    encod = self.tokenizer(text, max_length=self.max_len,
                           padding = 'max_length', truncation = True ,
                           return_tensors ='pt')
    encod = {key: encod[key].squeeze(0) for key in encod}
    encod['labels'] = torch.tensor(label, dtype=torch.long)
    return encod

In [ ]:
from torch.utils.data import DataLoader

train_dataset = MyDataset(tokenizer, df_tr['textt'], df_tr['topic'], max_len)
val_dataset = MyDataset(tokenizer, df_val['textt'], df_val['topic'], max_len)
test_dataset = MyDataset(tokenizer, df_te['textt'], df_te['topic'], max_len)


bs = 32
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=bs)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=bs)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=bs)


In [ ]:
import torch

print(train_dataset[4])

{'input_ids': tensor([   101,  26187,  38803,    130,  56576,   5063,  30272,  14564,  32919,
          3996,  11948,  38803,    130,  66062,   8215,   3590,    128,  96776,
         83737,    845,  18740,  18567,   6767,   6454,   6023,    845,  43086,
         55725,  17736,  31158,   6998,  20812,    128,   7603,  12993,  21759,
          1516,  33614,   2785,    146,   3422,  19932,  14710,    132,   5405,
          4721,    108,  11214,    130,  14544,    108,    128,   9976,    108,
        104234,  51561,    108,    128,  86351,  20741,   6454,   6023,   1469,
         44203,  10909,  86419,  78756,   9746,  31175,  13420,   7546,    851,
           869,  55909,   2259,    108,  27791,    106,  10249,    106,   9809,
           106,    108,  96776,  83737,  24425,  55725,    128,  89920,  41155,
           880,  54024,  26088,   6767,   2739,  19240,   1385,   2784,  58339,
           132,  20916,  27436,   8215,  29827,  86781,  19522,  16412,  10673,
           128,  32548,  1

## Досборка моей модели

Мы воспользуемся обычным bertом и дообучим в конце один слой для классификации:

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained('DeepPavlov/rubert-base-cased')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch.nn as nn

class Class(nn.Module):
  def __init__(self, bert ):
    super().__init__()
    self.bert=bert
    self.dropout = nn.Dropout(0.25)
    self.classify=nn.Linear(768, 5)
  def forward(self, ids, at_mask):
    ouputs = self.bert(ids, at_mask)
    last_hidden = ouputs.last_hidden_state
    cls = last_hidden[:, 0, :]
    x = self.dropout(cls)
    y = self.classify(x )
    return y

## Обучаемся

In [ ]:
bm_pathh = 'bm.pt'

In [ ]:
print(len(train_loader))

313


In [ ]:
best_acc=0
import torch.nn as nn
import torch.optim as optim

modell = Class(model).to('cuda')
optimizer = optim.AdamW(modell.parameters(), lr=3e-5)
criterion=nn.CrossEntropyLoss()

for epoch in range(6):
  modell.train()
  for idx, batch in enumerate(train_loader):
    ids, at_mask = batch['input_ids'].to('cuda'), batch['attention_mask'].to('cuda')
    label = batch['labels'].to('cuda')
    y_pred  = modell(ids, at_mask)
    optimizer.zero_grad()
    loss = criterion(y_pred, label)
    loss.backward()
    optimizer.step()

  alls, trues = len(df_val), 0
  modell.eval()
  with torch.no_grad():
    for batch in val_loader:
      ids, at_mask = batch['input_ids'].to('cuda'), batch['attention_mask'].to('cuda')
      label = batch['labels'].to('cuda')
      y_pred  = modell(ids, at_mask)
      preds = torch.argmax(y_pred, dim=1)
      trues += (preds==label).sum().item()
  print(f"AccuracyL {trues/alls*100:.1f} % in {epoch} epoch ")
  if trues/alls*100>best_acc:
    best_acc =  trues/alls*100
    torch.save(modell.state_dict(), bm_pathh )


AccuracyL 91.5 % in 0 epoch 
AccuracyL 93.8 % in 1 epoch 
AccuracyL 90.4 % in 2 epoch 
AccuracyL 90.9 % in 3 epoch 
AccuracyL 90.7 % in 4 epoch 
AccuracyL 91.1 % in 5 epoch 


In [ ]:
torch.save(modell.state_dict(), '/content/drive/MyDrive/best_model.pt')

## Тестим

In [ ]:
modell = Class(model).to('cuda' )
modell.load_state_dict(torch.load( bm_pathh ) )
torch.save(modell.state_dict(), '/content/drive/MyDrive/best_model.pt')
modell.eval()

Class(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tru

In [ ]:
alls, trues = len(df_val), 0
modell.eval()
with torch.no_grad():
  for batch in test_loader:
    ids, at_mask = batch['input_ids'].to('cuda'), batch['attention_mask'].to('cuda')
    label = batch['labels'].to('cuda')
    y_pred  = modell(ids, at_mask)
    preds = torch.argmax(y_pred, dim=1)
    trues += (preds==label).sum().item()
print(f"Accuracyy {trues/alls*100:.1f} % in test")

Accuracyy 91.5 % in test
